# Le Porte-Parole : sous-titres français en direct (démo)

Lance sur un GPU Colab (T4) le serveur de direct : S3 (encodeur Cohere + projecteur + Qwen2.5-1.5B) pour l'arabe, Opus-MT pour le texte gris provisoire, Qwen de S3 avec contexte pour le texte blanc définitif. La page `/live` s'ouvre sur le téléphone grâce au tunnel Cloudflare (HTTPS, obligatoire pour le micro).

**Avant de lancer** : Exécution > Modifier le type d'exécution > GPU T4. Dans les secrets Colab (clé, à gauche), créer `HF_TOKEN` avec accès au modèle gated `CohereLabs/cohere-transcribe-arabic-07-2026`, et activer l'accès au notebook.

Le dépôt est cloné depuis la branche `integration` de `SamiNakibETU/NLP`. Si `modeles/lora_traduction/` existe dans le dépôt, l'adaptateur LoRA est chargé automatiquement pour traduire (désactivé pour transcrire).

In [ ]:
!nvidia-smi -L
!apt-get -qq install -y ffmpeg > /dev/null
# Combinaison vérifiée sur T4 (Python 3.13, torch 2.11 cu128, transformers 5.20, pyannote.audio 4.0.7, speechbrain 1.1, bitsandbytes 0.50, piper-tts 1.8).
# numpy<2.3 : sinon numba (librosa) refuse de s'importer et Seamless/S3 ne lisent plus l'audio. pyannote en dernier : il peut toucher à torch.
!pip -q install fastapi "uvicorn[standard]" python-multipart "gradio>=6.0" "transformers>=5.4.0" accelerate peft bitsandbytes sentencepiece protobuf librosa soundfile pyyaml sacremoses trafilatura "brotli>=1.2" yt-dlp "qrcode[pil]" requests python-docx "numpy<2.3"
!pip -q install "speechbrain>=1.0" "piper-tts>=1.2" "edge-tts>=6.1"
!pip -q install "pyannote.audio>=4.0" "numpy<2.3"
!python -c "import torch, transformers, numpy; print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', numpy.__version__)"

In [ ]:
# Dépôt de l'équipe, branche integration (la mise à jour est un simple git pull si le dossier existe déjà)
import os, subprocess, sys
DEPOT = '/content/NLP'
if os.path.isdir(DEPOT):
    subprocess.run(['git', '-C', DEPOT, 'pull', '-q'], check=True)
else:
    subprocess.run(['git', 'clone', '-q', '-b', 'integration', 'https://github.com/SamiNakibETU/NLP', DEPOT], check=True)
APP = DEPOT + '/speech_app'
print('branche :', subprocess.run(['git', '-C', DEPOT, 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip())
print('projecteur S3 présent :', os.path.isfile(DEPOT + '/modeles/s3_temps1_50k/projecteur_meilleur.pt'))
LORA = DEPOT + '/modeles/lora_traduction'
print('adaptateur LoRA présent :', os.path.isfile(LORA + '/adapter_config.json'))

In [ ]:
# Réglages. Le jeton vient des secrets Colab : il n'est jamais écrit dans le notebook.
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
try:                                                      # facultatif : résumé de fin par l'API ScaleDown
    os.environ['SCALEDOWN_API_KEY'] = userdata.get('SCALEDOWN_API_KEY')
except Exception:
    os.environ.pop('SCALEDOWN_API_KEY', None)
print('Résumé ScaleDown :', 'oui' if os.environ.get('SCALEDOWN_API_KEY') else 'non (Qwen local)')
os.environ['PORTEPAROLE_NLP_REPO'] = DEPOT
os.environ['PORTEPAROLE_ENGINES'] = 's3,seamless,s0'          # moteurs du dépôt de fichier (/video, /api/translate) ; le direct n'en dépend pas
os.environ['PORTEPAROLE_MT'] = 'off'
os.environ['PORTEPAROLE_SESSIONS'] = '/content/sessions'  # audio + segments de chaque session (pour le rapport)
os.environ['PORTEPAROLE_API_TOKEN'] = ''                  # mettre un jeton pour protéger le serveur (le lien devra alors contenir ?token=...)
os.environ['PYTHONIOENCODING'] = 'utf-8'
if os.path.isfile(LORA + '/adapter_config.json'):
    os.environ['PORTEPAROLE_LORA'] = LORA
else:
    os.environ.pop('PORTEPAROLE_LORA', None)
print('LoRA :', os.environ.get('PORTEPAROLE_LORA', 'aucun (traduction par le Qwen de base)'))

In [ ]:
# Serveur en arrière-plan, puis tunnel public Cloudflare (sans compte)
import re, time, requests
serveur = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'api:app', '--host', '0.0.0.0', '--port', '7860'],
                           cwd=APP, stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        if requests.get('http://localhost:7860/api/health', timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
else:
    raise RuntimeError('le serveur ne répond pas : voir la dernière cellule (journal)')
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:7860'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
URL = None
for ligne in tunnel.stdout:
    if m := re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', ligne):
        URL = m.group(0)
        break
print('Serveur prêt :', URL)

In [ ]:
# Précharge S3 + Opus-MT (environ une minute la première fois), pour que la démo démarre sans attente
requests.post('http://localhost:7860/api/live/precharger').raise_for_status()
debut = time.time()
while time.time() - debut < 900:
    statut = requests.get('http://localhost:7860/api/live/statut').json()
    if statut['groupe_charge'] == 'direct':
        break
    time.sleep(3)
print('groupe chargé :', statut['groupe_charge'], '| ffmpeg :', statut['ffmpeg'], '| yt-dlp :', statut['yt_dlp'], '| LoRA :', statut['lora'])
print('chargement en %.0f s' % (time.time() - debut))

In [ ]:
# Lien et QR code à ouvrir sur le téléphone (devant la caméra)
import qrcode
from IPython.display import display, HTML
LIVE = URL + '/live'
display(HTML(f'<h3><a href="{LIVE}" target="_blank">{LIVE}</a></h3>'))
display(qrcode.make(LIVE).resize((320, 320)))
print('Sur le téléphone : ouvrir le lien, choisir Micro (HTTPS fourni par le tunnel) ou Lien (le serveur écoute YouTube / HLS).')
print('Plusieurs écrans : après le démarrage, "Copier le lien" donne /live?sid=... pour regarder la même session.')

In [ ]:
# Facultatif : démarrer une source URL depuis le notebook (ex. direct YouTube d'une chaîne d'information arabe)
# r = requests.post('http://localhost:7860/api/live/url', json={'url': 'https://www.youtube.com/watch?v=...'}); print(r.json())
# Puis ouvrir LIVE + '?sid=' + r.json()['sid'] sur les écrans à filmer.
# Fin de session : requests.post(f'http://localhost:7860/api/live/{sid}/arreter').json()  -> bilan (retard, réécriture)

In [ ]:
# Facultatif : « Vérifier avant de filmer » (3 à 4 minutes une fois les modèles en cache, 10 à 15 la première fois).
# Contrôle CE serveur avec de la vraie parole arabe (FLEURS) : direct par WebSocket, rapport de fin de session
# (locuteurs + relecture + résumé daté), lecture vocale. Affiche un bilan PASS/FAIL avec les chiffres.
# À lancer AVANT la prise : le rapport décharge le moteur de direct, le script le recharge ensuite (environ une minute).
!cd {APP} && python verifier_reel.py --rapide --serveur http://localhost:7860 --sortie /content/verification_reelle.json

In [ ]:
# Journal du serveur (en cas de souci)
!tail -n 40 /content/server.log